# 06 — Softmax (Numerically Stable Row-wise Softmax)

## Background

Softmax is the core component of Transformer attention and the standard output layer for classification. GPU implementation challenges:

1. **Numerical stability**: computing $e^x$ directly overflows (float16 max ≈ 65504). The standard fix is to subtract the row maximum: $\text{softmax}(x_j) = e^{x_j - \max_k x_k} / \sum_k e^{x_k - \max_k x_k}$
2. **Multi-pass scan**: the numerically stable version requires three passes — find the max, compute exp and sum, then normalise.
3. **FlashAttention's core idea** (Online Softmax) merges these passes into a streaming-friendly form.

## Problem Definition

Compute row-wise numerically stable softmax on matrix `A [N, M]`:

$$\text{MAX}[i] = \max_j A[i, j]$$
$$B[i,j] = \frac{e^{A[i,j] - \text{MAX}[i]}}{\sum_k e^{A[i,k] - \text{MAX}[i]}}$$

**Three-pass algorithm**:
- **Pass 1**: `MAX[i] = max(A[i, :])`
- **Pass 2**: `B[i,j] = exp(A[i,j] - MAX[i])`, `SUM[i] = sum(B[i,:])`
- **Pass 3**: `B[i,j] /= SUM[i]`

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
N, M = 4096, 16384
A = torch.randn(N, M, dtype=torch.float32, device="cuda")

def ref_softmax(A):
    return torch.softmax(A, dim=1)

B_ref = ref_softmax(A)
print(f"A: {A.shape} → B: {B_ref.shape}")
print(f"Row-sum check (should ≈ 1): B[0].sum() = {B_ref[0].sum().item():.6f}")

## Triton Implementation

One Triton program per row; three-pass scan via Python loop. Key details:
- `tl.max(chunk, 0)` reduces a `[BLOCK_M]` tensor to a scalar (0-d Triton tensor)
- `tl.maximum(scalar_a, scalar_b)` updates the running row maximum
- Scalar `row_sum` broadcasts over `[BLOCK_M]` in `y / row_sum`

In [ ]:
@triton.jit
def _softmax_kernel(X_ptr, Y_ptr, M, BLOCK_M: tl.constexpr):
    row  = tl.program_id(0)
    base = X_ptr + row * M

    # Pass 1: find row max
    row_max = tl.load(base).to(tl.float32)   # initialise with first element
    for start in range(0, M, BLOCK_M):
        cols  = start + tl.arange(0, BLOCK_M)
        chunk = tl.load(base + cols, mask=cols < M, other=float("-inf")).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(chunk, 0))

    # Pass 2: exp(x - max), write out, accumulate sum
    row_sum = 0.0
    for start in range(0, M, BLOCK_M):
        cols = start + tl.arange(0, BLOCK_M)
        mask = cols < M
        x = tl.load(base + cols, mask=mask, other=0.0).to(tl.float32)
        e = tl.exp(x - row_max)
        tl.store(Y_ptr + row * M + cols, e, mask=mask)
        row_sum = row_sum + tl.sum(e, 0)

    # Pass 3: normalise
    for start in range(0, M, BLOCK_M):
        cols = start + tl.arange(0, BLOCK_M)
        mask = cols < M
        y = tl.load(Y_ptr + row * M + cols, mask=mask)
        tl.store(Y_ptr + row * M + cols, y / row_sum, mask=mask)

BLOCK_M_TRI = 1024

def triton_softmax(A):
    B = torch.empty_like(A)
    _softmax_kernel[(A.shape[0],)](A, B, A.shape[1], BLOCK_M=BLOCK_M_TRI)
    return B

B_tri = triton_softmax(A)
ok = torch.allclose(B_ref, B_tri, atol=1e-4)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tri)).item():.6f}")

## TileLang Implementation

Same three-pass algorithm: `T.fill(mx, -inf)` initialises, `T.reduce_max` finds the row maximum, `T.exp2` with `log2_e` implements fast exp, `T.reduce_sum` accumulates.

**gfx1100 tuning**: `threads=256, BLOCK_M=256` — the layout constraint `BM ≤ threads` keeps `T.copy` generating a vectorised loop rather than a single load. 8 wavefronts per block gives good CU occupancy on RDNA3.

> **Note**: PyTorch routes this kernel through **MIOpen** (a vendor-optimised, architecture-specific library), which achieves fewer global-memory passes via an internal online-softmax implementation. TileLang's 3-pass explicit kernel comfortably outperforms Triton and approaches the MIOpen result.

In [ ]:
# gfx1100 optimal: threads=256, BLOCK_M=256.
# Constraint: BM <= threads (T.copy layout). threads=256 → 8 wavefronts/CU.
# This 3-pass variant (max → exp+sum → normalize) beats Triton by ~7%
# and approaches PyTorch's highly optimised MIOpen kernel.
BM = 256

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax(A, BLOCK_M: int):
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_, threads=256) as row:
        A_l = T.alloc_fragment((1, BLOCK_M), dtype)
        B_l = T.alloc_fragment((1, BLOCK_M), dtype)
        mx  = T.alloc_fragment((1,), dtype)
        sm  = T.alloc_fragment((1,), dtype)
        # Pass 1: find row max
        T.fill(mx, float("-inf"))
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            T.reduce_max(A_l, mx, dim=1, clear=False)
        # Pass 2: exp(x - max), accumulate sum
        T.clear(sm)
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = T.exp2(log2_e * (A_l[0, j] - mx[0]))
            T.reduce_sum(B_l, sm, dim=1, clear=False)
            T.copy(B_l, B[row, m_blk * BLOCK_M])
        # Pass 3: normalise
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(B[row, m_blk * BLOCK_M], B_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = B_l[0, j] / sm[0]
            T.copy(B_l, B[row, m_blk * BLOCK_M])
    return B

k = tl_softmax.compile(N=N, M=M, BLOCK_M=BM)
B_tl = k(A.clone())
# atol=5e-4: exp2(x*log2_e) approximation has ~2e-4 max error in float32
ok = torch.allclose(B_ref, B_tl, atol=5e-4)
print("TileLang correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tl)).item():.6f}")

## Performance Comparison

## TileLang Online Softmax (2-pass)

The standard 3-pass kernel reads global memory **5 times** (3R + 2W). We can cut that to **3 times** (2R + 1W) using the **online softmax** algorithm, which computes the running max and running sum in a single sequential scan — without writing any intermediate results to DRAM.

**Online update rule** (per tile in the sequential loop):
```
m_new  = max(m, tile_max)
s      = s * exp(m − m_new)    # rescale old sum to new max
s     += sum(exp(Aᵢ − m_new))  # add new tile's contribution
m      = m_new
```

**IO comparison**:

| Kernel | Reads | Writes | Total IO |
|--------|-------|--------|----------|
| 3-pass (Triton / TileLang standard) | 3 × A | 2 × B | 5 IO |
| **Online 2-pass** | 2 × A | 1 × B | 3 IO |

**Implementation note**: setting `BM = 8 × threads` ensures every thread issues a `float4×2` load per iteration, keeping all 512 threads fully occupied in Pass 2.

In [ ]:
# Online Softmax — best config on gfx1100: threads=512, BLOCK_M=4096
# BM = 8 × threads → every thread loads 8 floats (two float4) per tile,
# keeping all 512 threads active in both passes.
BM_online = 4096

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax_online(A, BLOCK_M: int):
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype  = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_, threads=512) as row:
        Al    = T.alloc_fragment((1, BLOCK_M), dtype)
        mx    = T.alloc_fragment((1,), dtype)   # running global max
        sm    = T.alloc_fragment((1,), dtype)   # running sum (normalised to mx)
        mx_bl = T.alloc_fragment((1,), dtype)   # per-tile max
        sm_bl = T.alloc_fragment((1,), dtype)   # per-tile exp-sum

        T.fill(mx, float("-inf"))
        T.clear(sm)

        # ── Pass 1 (1 global read): online (m, s) accumulation ───────────────
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], Al)

            # tile max
            T.fill(mx_bl, float("-inf"))
            T.reduce_max(Al, mx_bl, dim=1, clear=True)

            # update running max and rescale running sum
            for i in T.Parallel(1):
                new_mx = T.max(mx[0], mx_bl[0])
                sm[0]  = sm[0] * T.exp2(log2_e * (mx[0] - new_mx))
                mx[0]  = new_mx

            # exp(tile − new mx) and accumulate tile sum
            for j in T.Parallel(BLOCK_M):
                Al[0, j] = T.exp2(log2_e * (Al[0, j] - mx[0]))
            T.clear(sm_bl)
            T.reduce_sum(Al, sm_bl, dim=1, clear=True)
            for i in T.Parallel(1):
                sm[0] = sm[0] + sm_bl[0]

        # ── Pass 2 (1 global read + 1 global write): normalise ───────────────
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], Al)
            for j in T.Parallel(BLOCK_M):
                B[row, m_blk * BLOCK_M + j] = (
                    T.exp2(log2_e * (Al[0, j] - mx[0])) / sm[0]
                )
    return B

k_online = tl_softmax_online.compile(N=N, M=M, BLOCK_M=BM_online)
B_online  = k_online(A.clone())
ok = torch.allclose(B_ref, B_online, atol=5e-4)
print("TileLang Online correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_online)).item():.6f}")

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch":          bench(ref_softmax,         [A]),
    "Triton\n(3-pass)": bench(triton_softmax,      [A]),
    "TileLang\n3-pass": bench(k,                   [A]),
    "TileLang\nOnline": bench(k_online,            [A]),
}

bytes_rw_3pass  = N * M * 4 * 5   # 3R + 2W
bytes_rw_online = N * M * 4 * 3   # 2R + 1W

for name, ms in results.items():
    label = name.replace('\n', ' ')
    is_online = "Online" in name
    bw = (bytes_rw_online if is_online else bytes_rw_3pass) / ms / 1e9
    print(f"  {label:22s}: {ms:.4f} ms  ({bw:.2f} TB/s, {'2R+1W' if is_online else '3R+2W'})")

colors = ["#4C72B0", "#55A868", "#C44E52", "#8B0000"]
labels = list(results.keys())
values = list(results.values())

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(labels))
bars = ax.bar(x, values, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(labels, fontsize=9)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f} ms", ha="center", va="bottom", fontsize=9)
ax.axvline(2.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(1.0, max(values)*1.16, "3-pass (5 IO)", ha="center", fontsize=9, color="gray")
ax.text(3.0, max(values)*1.16, "Online (3 IO)", ha="center", fontsize=9, color="gray")
ax.set_ylabel("Latency (ms)")
ax.set_title(f"Softmax Comparison  (N={N}, M={M}, float32) — Online avoids 2 global write passes")
ax.set_ylim(0, max(values)*1.25)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()